# 06 · Benchmark: which systems find the examiner's prior art?

The main experiment of the study. For every test case (the first claim as filed), each system ranks the 17,622 corpus patents, and the ranking is scored against the citations:

| System | Type |
|---|---|
| BM25 (passages, MaxP) and BM25 (whole patent) | Keyword search |
| `bge-small-en-v1.5` | Small general embedder (the current demo) |
| `PaECTER` | Patent-specific (citation-trained, BERT-for-Patents) |
| `patembed-base` | Patent-specific (PatenTEB) |
| `Qwen3-Embedding-0.6B` | General LLM-based embedder |

**Metrics:** Recall@10/50/100, MAP@100, nDCG@10, MRR, PRES@100 (Magdy & Jones, 2010), and reading effort (documents read to reach 80% recall). Each is reported for all citations, **examiner-only** and **applicant-only** citations, and for prior art from the **same** vs **other** applicants. It also measures how often each system's top 10 come from the query's own applicant.

**Runtime → Change runtime type → T4 GPU** before running. Passage embeddings are saved to Drive in parts, so if Colab disconnects, just run all again: finished work is reused. The whole notebook takes roughly 45-75 minutes the first time.

In [ ]:
BRANCH = "v2-corpus"
!rm -rf /content/app
!git clone -q --depth 1 -b {BRANCH} https://github.com/Renuk9390/patent-prior-art-explorer.git /content/app
!pip install -q -U "sentence-transformers>=3.0" bm25s PyStemmer
import sys; sys.path.insert(0, "/content/app/src")

from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/patent-search"

import os, time, numpy as np, pandas as pd, torch
from patentsearch.eval.benchmark import evaluate_run, maxp, same_applicant_share, summarize
from patentsearch.eval.qrels import assignee_set

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(no GPU: switch the runtime to T4 GPU)")

EMB_DIR, RUN_DIR, METRICS_DIR = f"{DRIVE_DIR}/embeddings", f"{DRIVE_DIR}/runs", f"{DRIVE_DIR}/metrics"
for d in (EMB_DIR, RUN_DIR, METRICS_DIR):
    os.makedirs(d, exist_ok=True)

passages = pd.read_parquet(f"{DRIVE_DIR}/passages.parquet")
patents = pd.read_parquet(f"{DRIVE_DIR}/patents.parquet")
testset = pd.read_parquet(f"{DRIVE_DIR}/testset.parquet")
qrels = pd.read_parquet(f"{DRIVE_DIR}/qrels.parquet")
testset = testset[testset["publication_number"].isin(set(qrels["query_id"]))].reset_index(drop=True)

P_TEXT = passages["text"].tolist()
P_DOC = passages["publication_number"].to_numpy()
P_LABEL = passages["section"].to_numpy()
Q_IDS = testset["publication_number"].tolist()
Q_TEXT = testset["query"].tolist()
print(f"{len(P_TEXT):,} passages from {len(patents):,} patents | {len(Q_IDS):,} test queries | {len(qrels):,} labels")

## 1 · Systems

In [ ]:
MODELS = {
    "bge-small": dict(hf="BAAI/bge-small-en-v1.5", batch=256,
                      query_prompt="Represent this sentence for searching relevant passages: ", doc_prompt=""),
    "paecter": dict(hf="mpi-inno-comp/paecter", batch=64, query_prompt="", doc_prompt=""),
    "patembed-base": dict(hf="datalyes/patembed-base", batch=64,
                          query_prompt="encode query for same document retrieval: ",
                          doc_prompt="encode document for same retrieval: "),
    "qwen3-0.6b": dict(hf="Qwen/Qwen3-Embedding-0.6B", batch=32,
                       query_prompt="Instruct: Given a patent claim, retrieve patent passages that disclose the claimed features\nQuery:",
                       doc_prompt=""),
}
RUN_MODELS = list(MODELS)          # or e.g. ["bge-small"] to run one model
MAX_SEQ = 512                      # passages are <=250 words; queries truncated beyond this
SHARD = 40_000                     # passage embeddings saved to Drive in parts of this size
TOP_PASSAGES, TOP_DOCS = 3000, 100

## 2 · Dense systems: embed, search, save

In [ ]:
from sentence_transformers import SentenceTransformer

def load_model(cfg):
    model = SentenceTransformer(
        cfg["hf"], device=DEVICE,
        model_kwargs={"torch_dtype": torch.float16} if DEVICE == "cuda" else {},
        prompts={"query": cfg["query_prompt"], "document": cfg["doc_prompt"]},
    )
    model.max_seq_length = MAX_SEQ
    return model

def encode(model, texts, prompt, batch):
    return model.encode(texts, batch_size=batch, prompt=prompt or None, normalize_embeddings=True,
                        convert_to_numpy=True, show_progress_bar=False).astype(np.float16)

def passage_embeddings(name, cfg):
    path = f"{EMB_DIR}/{name}.npy"
    if os.path.exists(path):
        return np.load(path), None
    model, parts, t0 = load_model(cfg), [], time.time()
    for start in range(0, len(P_TEXT), SHARD):
        part_path = f"{EMB_DIR}/{name}.part{start // SHARD:02d}.npy"
        if not os.path.exists(part_path):
            np.save(part_path, encode(model, P_TEXT[start:start + SHARD], cfg["doc_prompt"], cfg["batch"]))
            print(f"  {name}: passages {start:,}-{min(start + SHARD, len(P_TEXT)):,} done ({time.time() - t0:.0f}s)")
        parts.append(np.load(part_path))
    E = np.concatenate(parts)
    np.save(path, E)
    for start in range(0, len(P_TEXT), SHARD):
        os.remove(f"{EMB_DIR}/{name}.part{start // SHARD:02d}.npy")
    return E, time.time() - t0

def search(E_p, E_q):
    P = torch.from_numpy(E_p).to(DEVICE)
    P = P if DEVICE == "cuda" else P.float()
    run, best = {}, {}
    for i in range(0, len(E_q), 256):
        Q = torch.from_numpy(E_q[i:i + 256]).to(DEVICE)
        Q = Q if DEVICE == "cuda" else Q.float()
        top = torch.topk(Q @ P.T, min(TOP_PASSAGES, P.shape[0]), dim=1).indices.cpu().numpy()
        for j, row in enumerate(top):
            docs, bp = maxp(row, P_DOC, TOP_DOCS)
            run[Q_IDS[i + j]], best[Q_IDS[i + j]] = docs, bp
    return run, best

def save_run(name, run, best):
    rows = [(q, r + 1, d, P_LABEL[best[q][r]] if best else "") for q, ds in run.items() for r, d in enumerate(ds)]
    pd.DataFrame(rows, columns=["query_id", "rank", "doc_id", "best_section"]).to_parquet(f"{RUN_DIR}/{name}.parquet", index=False)

timings = {}
for name in RUN_MODELS:
    if os.path.exists(f"{RUN_DIR}/{name}.parquet"):
        print(f"{name}: run already saved, skipping"); continue
    cfg = MODELS[name]
    print(f"{name}: embedding {len(P_TEXT):,} passages ...")
    E_p, secs = passage_embeddings(name, cfg)
    model = load_model(cfg)
    t0 = time.time()
    E_q = encode(model, Q_TEXT, cfg["query_prompt"], cfg["batch"])
    q_secs = time.time() - t0
    run, best = search(E_p, E_q)
    save_run(name, run, best)
    timings[name] = {"passage_encoding_s": secs, "queries_per_s": len(Q_TEXT) / q_secs, "dim": E_p.shape[1]}
    print(f"{name}: done. {timings[name]}")
    del model, E_p; torch.cuda.empty_cache() if DEVICE == "cuda" else None

## 3 · BM25 (keyword search)

In [ ]:
import bm25s, Stemmer
stemmer = Stemmer.Stemmer("english")
q_tok = bm25s.tokenize(Q_TEXT, stopwords="en", stemmer=stemmer, show_progress=False)

if not os.path.exists(f"{RUN_DIR}/bm25-passages.parquet"):
    t0 = time.time()
    bm = bm25s.BM25(); bm.index(bm25s.tokenize(P_TEXT, stopwords="en", stemmer=stemmer, show_progress=False), show_progress=False)
    idx, _ = bm.retrieve(q_tok, k=min(TOP_PASSAGES, len(P_TEXT)), show_progress=False, n_threads=-1)
    run, best = {}, {}
    for qid, row in zip(Q_IDS, idx):
        run[qid], best[qid] = maxp(row, P_DOC, TOP_DOCS)
    save_run("bm25-passages", run, best)
    print(f"bm25-passages: {time.time() - t0:.0f}s")

if not os.path.exists(f"{RUN_DIR}/bm25-document.parquet"):
    t0 = time.time()
    doc_text = passages.groupby("publication_number")["text"].apply(" ".join)
    doc_ids = doc_text.index.to_numpy()
    bm = bm25s.BM25(); bm.index(bm25s.tokenize(doc_text.tolist(), stopwords="en", stemmer=stemmer, show_progress=False), show_progress=False)
    idx, _ = bm.retrieve(q_tok, k=min(TOP_DOCS, len(doc_ids)), show_progress=False, n_threads=-1)
    run = {qid: [doc_ids[i] for i in row] for qid, row in zip(Q_IDS, idx)}
    save_run("bm25-document", run, None)
    print(f"bm25-document: {time.time() - t0:.0f}s")

## 4 · Results

In [ ]:
runs = {}
for f in sorted(os.listdir(RUN_DIR)):
    if f.endswith(".parquet"):
        r = pd.read_parquet(f"{RUN_DIR}/{f}").sort_values(["query_id", "rank"])
        runs[f[:-8]] = r.groupby("query_id")["doc_id"].apply(list).to_dict()

qa = {r.publication_number: assignee_set(r.assignees) for r in testset.itertuples()}
da = {r.publication_number: assignee_set(r.assignees) for r in patents.itertuples()}

results = {}
for name, run in runs.items():
    pq = evaluate_run(run, qrels)
    pq.to_parquet(f"{METRICS_DIR}/{name}.parquet", index=False)
    s = summarize(pq)
    s["top10_same_applicant"] = same_applicant_share(run, qa, da, 10)
    results[name] = s
table = pd.DataFrame(results).T
table.to_csv(f"{DRIVE_DIR}/benchmark_summary.csv")
pd.set_option("display.precision", 3)

print("=== Standard metrics (all citations) ===")
print(table[["R@10", "R@50", "R@100", "MAP@100", "nDCG@10", "MRR", "PRES@100"]].sort_values("PRES@100", ascending=False).to_string())
print("\n=== Reading effort: documents read to find 80% of the cited prior art (top 100) ===")
print(table[["effort@80_reached", "effort@80_median"]].to_string())

In [ ]:
print("=== A · Examiner-added vs applicant-cited prior art ===")
cols = ["examiner_R@10", "applicant_R@10", "examiner_R@100", "applicant_R@100", "examiner_PRES@100", "applicant_PRES@100"]
print(table[cols].to_string())
print(f"\nPaired (same {int(table['paired_n'].iloc[0])} queries with both kinds):")
print(table[["paired_examiner_R@100", "paired_applicant_R@100"]].to_string())

print("\n=== B · Portfolio bias ===")
ref = qrels.groupby("source")["same_applicant"].mean()
print("Share of cited prior art from the query's own applicant:  "
      f"examiner {ref.get('examiner', float('nan')):.3f} | applicant {ref.get('applicant', float('nan')):.3f} | all {qrels['same_applicant'].mean():.3f}")
print(table[["top10_same_applicant", "same_app_R@100", "diff_app_R@100"]].to_string())

In [ ]:
# Where does the matching evidence sit? Section of the best passage for retrieved cited patents
rows = []
for name in runs:
    r = pd.read_parquet(f"{RUN_DIR}/{name}.parquet")
    if (r["best_section"] == "").all():
        continue
    hits = r.merge(qrels, on=["query_id", "doc_id"])
    hits = hits[hits["rank"] <= 100]
    rows.append(hits["best_section"].value_counts(normalize=True).rename(name))
print("=== Section of the best-matching passage, for cited patents found in the top 100 ===")
print(pd.DataFrame(rows).round(3).to_string())
if timings:
    print("\n=== Speed (this session, GPU) ===")
    print(pd.DataFrame(timings).T.round(1).to_string())